# 1 · Modelo de referencia

**Proyecto:** Predicción de readmisión hospitalaria en pacientes diabéticos
**Dataset:** *Diabetes 130-US hospitals for years 1999-2008* (UCI Machine Learning Repository)
**Autor:** Diego Rodríguez Díaz del Campo

---

### Objetivo de este notebook

Fijar las reglas con las que se compararán todos los modelos del proyecto, es decir, cómo se valida y qué
nota hay que superar, y obtener el primer resultado con un modelo simple: la regresión logística.

El modelo debe **ordenar los ingresos por riesgo de readmisión a 30 días** en el momento del alta, para
priorizar el seguimiento. Se mide con **PR-AUC** como métrica principal, **ROC-AUC** como complementaria y
recall y precisión en el **20 % de ingresos de mayor riesgo**.

| | |
|---|---|
| **Entrada** | `data/train.csv` (79.433 × 160), salida del notebook 04 de [readmision_diabetes_eda](https://github.com/diieeguiitooo/readmision_diabetes_eda) |
| **Variable objetivo** | `readmitted_30d` (1 = reingreso antes de 30 días) |
| **Salida** | Resultados de validación de la regresión logística: la referencia para los notebooks 02 y 03 |

`data/test.csv` no se usa en este notebook: queda reservado para la evaluación final (notebook 03).

### Contenido

1. Carga y comprobaciones
2. Separación de predictoras, objetivo y grupos
3. Diseño de la validación
4. Referencias que hay que superar
5. Regresión logística
6. Conclusiones

## 1 · Carga y comprobaciones

Se parte de `data/train.csv`, descargado con `scripts/descargar_datos.py` desde un commit fijo de
`readmision_diabetes_eda`. El script ya garantiza que el archivo es idéntico byte a byte al que guardó el
notebook 04 (huella SHA-256); lo que falta comprobar es que pandas lo ha leído como se espera.

#### Decisiones

- **Solo se carga `train.csv`.** `test.csv` no se abre hasta el notebook 03: cualquier decisión tomada
  mirándolo lo convertiría en parte del entrenamiento, y su nota final dejaría de medir el rendimiento con
  datos nuevos.
- **`read_csv` sin opciones.** Las 160 columnas son enteros, así que no reaparece el problema de tipos de los
  `*_id` ni el aviso de `payer_code` del proyecto EDA.

### Qué se comprueba

Cada comprobación se compara con el valor que dejó el notebook 04:

| Comprobación | Esperado | Qué descarta |
|---|---|---|
| Dimensiones | 79.433 × 160 | Filas perdidas o una columna de índice añadida (`Unnamed: 0`) |
| Tipos | 160 columnas `int64` | Una columna leída como texto |
| Nulos | 0 | Valores vacíos, que la regresión logística no admite |
| Objetivo | 70.362 ceros y 9.071 unos (11,42 %) | Un objetivo distinto del preparado |
| Pacientes distintos | 55.992 | Grupos que no cuadran con la partición del notebook 04 |

In [1]:
# Este apartado solo carga y comprueba el CSV: basta con pandas y pathlib.
# scikit-learn se importa en cada apartado donde se usa por primera vez.
import pandas as pd
from pathlib import Path

# El notebook se ejecuta desde notebooks/, así que la raíz del proyecto es la carpeta padre.
# Construir las rutas desde ahí evita depender de dónde esté clonado el repo.
BASE = Path.cwd().parent
DATOS = BASE / 'data'
RUTA_TRAIN = DATOS / 'train.csv'

# Si el archivo no está, falla aquí con un mensaje claro y no más adelante dentro de read_csv
assert RUTA_TRAIN.exists(), f'No se encuentra {RUTA_TRAIN}: ejecuta scripts/descargar_datos.py'
print(f'Raíz del proyecto: {BASE}')

Raíz del proyecto: /Users/diegorodriguezdiazdelcampo/Cosas Diego/IA/Proyectos personales/Model Diabetes


In [2]:
df = pd.read_csv(RUTA_TRAIN)

# Las cinco comprobaciones de la tabla, en el mismo orden
print(f'Dimensiones:          {df.shape}')
print(f'Tipos:                {df.dtypes.astype(str).value_counts().to_dict()}')
print(f'Nulos:                {df.isna().sum().sum()}')

# Recuento por clase, no solo cuántas clases hay: así se comparan con los 70.362 / 9.071 del NB04
conteo_objetivo = df['readmitted_30d'].value_counts().sort_index()
print(f'Objetivo (0 / 1):     {conteo_objetivo[0]} / {conteo_objetivo[1]} '
      f'({df["readmitted_30d"].mean():.2%} positivos)')
print(f'Pacientes distintos:  {df["patient_nbr"].nunique()}')

Dimensiones:          (79433, 160)
Tipos:                {'int64': 160}
Nulos:                0
Objetivo (0 / 1):     70362 / 9071 (11.42% positivos)
Pacientes distintos:  55992


### Interpretación del apartado 1

Las cinco comprobaciones coinciden con lo que dejó el notebook 04:

| Comprobación | Esperado | Obtenido |
|---|---|---|
| Dimensiones | 79.433 × 160 | 79.433 × 160 |
| Tipos | 160 columnas `int64` | 160 columnas `int64` |
| Nulos | 0 | 0 |
| Objetivo | 70.362 ceros y 9.071 unos (11,42 %) | 70.362 ceros y 9.071 unos (11,42 %) |
| Pacientes distintos | 55.992 | 55.992 |

Las comprobaciones de tipos y de nulos no son independientes: una columna `int64` no puede contener
`NaN`, que es un valor decimal, así que 160 columnas enteras implican 0 nulos. Se mantienen las dos
porque la de nulos se lee sin conocer esa regla.

El archivo se ha leído como se esperaba y se puede pasar a separar sus columnas.

## 2 · Separación de predictoras, objetivo y grupos

scikit-learn recibe por separado la matriz de predictoras (`X`) y el vector objetivo (`y`). La
validación agrupada del apartado 3 necesita además un tercer vector, `grupos`, que indica a qué
paciente pertenece cada ingreso.

#### Decisiones

- **`y` es `readmitted_30d`.** Está en la posición 12, no al final, así que se selecciona por nombre y
  no por posición.
- **`patient_nbr` sale de `X` y se guarda como `grupos`.** Un identificador no tiene significado
  clínico y el modelo podría aprender patrones espurios a partir del número. Solo sirve para que todos
  los ingresos de un mismo paciente caigan en el mismo pliegue.
- **`X` son las 158 columnas restantes, sin escalar.** El escalado va dentro del pipeline (apartado 5)
  y se aprende solo con los datos de entrenamiento de cada pliegue.

### Qué se comprueba

| Comprobación | Esperado |
|---|---|
| Dimensiones de `X` | 79.433 × 158 |
| Longitud de `y` y de `grupos` | 79.433 cada uno |
| `readmitted_30d` y `patient_nbr` en `X` | Ninguna de las dos |

In [3]:
# Objetivo y grupos se seleccionan por nombre; X es todo lo demás
X = df.drop(columns=['readmitted_30d', 'patient_nbr'])
y = df['readmitted_30d']
grupos = df['patient_nbr']

# Las tres comprobaciones de la tabla
print(f'Dimensiones de X:     {X.shape}')
print(f'Longitud de y:        {len(y)}')
print(f'Longitud de grupos:   {len(grupos)}')
print(f'Objetivo en X:        {"readmitted_30d" in X.columns}')
print(f'patient_nbr en X:     {"patient_nbr" in X.columns}')

Dimensiones de X:     (79433, 158)
Longitud de y:        79433
Longitud de grupos:   79433
Objetivo en X:        False
patient_nbr en X:     False


### Interpretación del apartado 2

| Comprobación | Esperado | Obtenido |
|---|---|---|
| Dimensiones de `X` | 79.433 × 158 | 79.433 × 158 |
| Longitud de `y` y de `grupos` | 79.433 cada uno | 79.433 y 79.433 |
| `readmitted_30d` y `patient_nbr` en `X` | Ninguna de las dos | Ninguna (`False` y `False`) |

`X`, `y` y `grupos` tienen las mismas filas y en el mismo orden, porque los tres salen de `df` sin
reordenar. El ingreso de la fila *i* de `X` tiene su objetivo en la fila *i* de `y` y su paciente en la
fila *i* de `grupos`, que es lo que necesita la validación del apartado 3.

## 3 · Diseño de la validación

`test` queda reservado para la evaluación final, así que los modelos se comparan con validación cruzada
sobre `train`: se divide en 5 pliegues y, en cada vuelta, se entrena un modelo nuevo con cuatro y se evalúa
con el quinto. Todos los modelos del proyecto se medirán con este mismo reparto, para que sus notas sean
comparables.

#### Decisiones

- **Validación agrupada por paciente.** Hay 55.992 pacientes para 79.433 ingresos. Si los ingresos de un
  mismo paciente quedaran a ambos lados, el modelo se evaluaría con pacientes que ya conoce y la nota saldría
  optimista.
- **Estratificada: `StratifiedGroupKFold` y no `GroupKFold`.** El azar de la PR-AUC es la proporción de
  positivos, así que un pliegue con más positivos tiene un suelo más alto. Con `GroupKFold`, la proporción
  de positivos de los pliegues de validación variaba entre el 10,76 % y el 11,91 %; estratificando, los
  cinco quedan en el 11,42 %.
- **5 pliegues.** Cada validación tiene unos 1.810 positivos, suficientes para que su PR-AUC sea estable;
  cada modelo entrena con el 80 % de `train`, y el coste se mantiene asumible para el notebook 02, que
  repetirá la validación con varios modelos e hiperparámetros.
- **`shuffle=True` con `random_state=0`.** Sin barajar, el reparto sigue el orden de `patient_nbr`, que se
  relaciona con `number_diagnoses` (correlación de 0,225). Los pliegues salen igual de equilibrados en los
  dos casos; se baraja para poder repetir la validación con otras semillas si hay que desempatar modelos. La
  semilla se fija antes de ver ningún resultado y no se cambia.

### Qué se comprueba

| Comprobación | Esperado | Qué descarta |
|---|---|---|
| Pacientes compartidos entre entrenamiento y validación | 0 en cada vuelta | Fuga de pacientes |
| Positivos en cada pliegue de validación | 11,42 % | Pliegues con suelos de PR-AUC distintos |
| Ingresos en cada pliegue de validación | ≈ 15.887 (79.433 / 5) | Pliegues descompensados |
| Ingresos en validación, sumando los 5 pliegues | 79.433, sin repetir ninguno | Ingresos sin evaluar o evaluados dos veces |

In [4]:
from sklearn.model_selection import StratifiedGroupKFold

# Reparto en 5 pliegues por paciente, estratificado por el objetivo y con semilla fija
divisor = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

# Se recorren las 5 vueltas y se guardan las comprobaciones de cada una; las posiciones de
# validación se acumulan para comprobar al final que cada ingreso se valida una sola vez
filas = []
todas_valid = []
for vuelta, (pos_entreno, pos_valid) in enumerate(divisor.split(X, y, grupos), start=1):
    compartidos = set(grupos.iloc[pos_entreno]) & set(grupos.iloc[pos_valid])
    filas.append({
        'vuelta': vuelta,
        'pacientes_compartidos': len(compartidos),
        'positivos_valid': y.iloc[pos_valid].mean(),
        'ingresos_valid': len(pos_valid),
    })
    todas_valid.extend(pos_valid)

comprobaciones = pd.DataFrame(filas).set_index('vuelta')
display(comprobaciones.style.format({'positivos_valid': '{:.2%}', 'ingresos_valid': '{:,}'}))
print(f'Ingresos en validación, sumando los 5 pliegues: {len(todas_valid)}')
print(f'Ingresos distintos entre ellos:                 {len(set(todas_valid))}')

,pacientes_compartidos,positivos_valid,ingresos_valid
vuelta,,,
1,0,11.42%,"15,887"
2,0,11.42%,"15,886"
3,0,11.42%,"15,888"
4,0,11.43%,"15,886"
5,0,11.42%,"15,886"


Ingresos en validación, sumando los 5 pliegues: 79433
Ingresos distintos entre ellos:                 79433


### Interpretación del apartado 3

| Comprobación | Esperado | Obtenido |
|---|---|---|
| Pacientes compartidos entre entrenamiento y validación | 0 en cada vuelta | 0 en las 5 vueltas |
| Positivos en cada pliegue de validación | 11,42 % | 11,42 % en los 5 (entre 11,42 % y 11,43 %) |
| Ingresos en cada pliegue de validación | ≈ 15.887 | Entre 15.886 y 15.888 |
| Ingresos en validación, sumando los 5 pliegues | 79.433, sin repetir ninguno | 79.433, todos distintos |

El reparto cumple las cuatro condiciones. Ningún paciente se evalúa con ingresos que el modelo ya ha visto,
los cinco pliegues tienen el mismo suelo de PR-AUC y cada ingreso de `train` se valida exactamente una vez.
`divisor` queda fijado y es el que usarán todos los modelos del proyecto.

## 4 · Referencias que hay que superar

Una nota solo tiene sentido comparada con lo que se obtiene sin modelo. Se fijan dos referencias: el
**azar**, que marca el suelo (un modelo por debajo no ha aprendido nada), y una **regla simple** con
`number_inpatient`, que marca el listón (un modelo que no la supera no justifica su complejidad).

#### Decisiones

- **Azar: valores teóricos, sin código.** Ordenar al azar da ROC-AUC 0,5; la precisión se queda en la
  proporción de positivos en todo el recorrido, así que la PR-AUC es 0,1142; marcar al azar el 20 % de los
  ingresos captura el 20 % de los reingresos, con precisión 0,1142.
- **Regla: ordenar por `number_inpatient`.** Es la variable más asociada a la readmisión en el EDA y la regla
  más fácil de aplicar sin modelo: priorizar a quien más ingresos ha tenido el año anterior. No se entrena
  nada: el propio valor de la variable sirve de puntuación.
- **Empates resueltos por sorteo con semilla fija.** El 66,7 % de los ingresos tiene 0 ingresos previos y el
  corte del 20 % cae dentro del grupo con 1, así que la regla no decide quién entra. Se sortea entre los
  empatados para tener una lista con un puesto por ingreso, la que usaría un hospital que aplicase la regla.
  Desempatar con otra variable crearía una regla distinta; desempatar por orden de fila dependería de cómo
  está ordenado el CSV.
- **La PR-AUC se calcula también con el sorteo.** El sorteo no cambia el orden entre grupos, pero sí la
  PR-AUC: sin él, `average_precision_score` trata cada bloque empatado como un solo escalón con la precisión
  del final del bloque. Con el sorteo, la regla se mide sobre una lista sin empates, igual que la regresión
  logística del apartado 5.
- **Misma validación que los modelos.** La regla se mide en los 5 pliegues de validación de `divisor` y se
  da la media, para que su nota sea comparable con la de la regresión logística del apartado 5.

### Qué se comprueba

| Referencia | PR-AUC | ROC-AUC | Recall en el 20 % |
|---|---|---|---|
| Azar | 0,1142 | 0,5 | 0,20 |
| Regla `number_inpatient` | Por encima del azar | Por encima del azar | Por encima del azar |

In [5]:
import numpy as np

# Generador con semilla fija: el sorteo de los empates sale igual en cada ejecución
generador = np.random.default_rng(0)

# Un decimal al azar por ingreso, entre 0 y 0,9. Al ser menor que 1, solo desempata entre ingresos
# con el mismo number_inpatient y nunca cambia el orden de la regla
ruido = generador.uniform(0, 0.9, size=len(X))
puntuacion_regla = X['number_inpatient'] + ruido

# Comprobación en todas las filas: quitando el decimal, la puntuación es number_inpatient
print(f'Parte entera igual a number_inpatient: {(puntuacion_regla.astype(int) == X["number_inpatient"]).all()}')

# Rango de la puntuación para los primeros valores de number_inpatient
tabla = pd.DataFrame({'number_inpatient': X['number_inpatient'], 'puntuacion_regla': puntuacion_regla})
display(tabla.groupby('number_inpatient')['puntuacion_regla'].agg(['min', 'max', 'count']).head(4))


Parte entera igual a number_inpatient: True


,min,max,count
number_inpatient,,,
0,0.000003,0.899988,52988
1,1.000029,1.899997,15100
2,2.000021,2.899972,5819
3,3.000083,3.899579,2624


In [6]:
from sklearn.metrics import average_precision_score, roc_auc_score

# La regla no se entrena: en cada vuelta solo se mide en el pliegue de validación, el mismo con el
# que se evaluarán los modelos, para poder compararlos pliegue a pliegue
filas_regla = []
for vuelta, (pos_entreno, pos_valid) in enumerate(divisor.split(X, y, grupos), start=1):
    y_valid = y.iloc[pos_valid]
    puntuacion_valid = puntuacion_regla.iloc[pos_valid]

    # Recall en el 20 %: se llama al 20 % del pliegue con la puntuación más alta y se cuenta qué
    # parte de sus reingresos cae entre ellos (.loc, porque nlargest devuelve etiquetas)
    k = int(0.2 * len(y_valid))
    llamados = puntuacion_valid.nlargest(k).index
    recall_20 = y_valid.loc[llamados].sum() / y_valid.sum()

    filas_regla.append({
        'vuelta': vuelta,
        'pr_auc': average_precision_score(y_valid, puntuacion_valid),
        # Sin sorteo, solo para ver cuánto pesa la forma de contar los empates; no se usa para comparar
        'pr_auc_sin_sorteo': average_precision_score(y_valid, X['number_inpatient'].iloc[pos_valid]),
        'roc_auc': roc_auc_score(y_valid, puntuacion_valid),
        'recall_20': recall_20,
    })

# Tabla por pliegue, con la media y la desviación debajo: la media es la nota que se compara con el azar
resultados_regla = pd.DataFrame(filas_regla).set_index('vuelta')
resumen_regla = resultados_regla.agg(['mean', 'std']).rename(index={'mean': 'media', 'std': 'desv.'})
display(pd.concat([resultados_regla, resumen_regla]).style.format('{:.4f}'))


,pr_auc,pr_auc_sin_sorteo,roc_auc,recall_20
1,0.2039,0.1845,0.6086,0.3528
2,0.2001,0.1804,0.6200,0.3539
3,0.2013,0.1813,0.6081,0.3501
4,0.1975,0.1783,0.6217,0.3625
5,0.1863,0.1691,0.6041,0.3286
media,0.1978,0.1787,0.6125,0.3496
desv.,0.0068,0.0058,0.0078,0.0126


### Interpretación del apartado 4

| Referencia | PR-AUC | ROC-AUC | Recall en el 20 % |
|---|---|---|---|
| Azar | 0,1142 | 0,5 | 0,20 |
| Regla `number_inpatient` (media ± desv.) | 0,1978 ± 0,0068 | 0,6125 ± 0,0078 | 0,3496 ± 0,0126 |

La regla supera al azar en las tres métricas y en los cinco pliegues, también en el peor (PR-AUC 0,1863,
recall 0,3286). Llamando al 20 % de los ingresos con más ingresos previos se captura el 35 % de los
reingresos, frente al 20 % del azar: 1,5 reingresos más de cada 10. La ROC-AUC de 0,61 indica que la
separación existe pero es modesta.

Sin el sorteo, la PR-AUC de la regla baja a 0,1787 ± 0,0058, porque los empates se cuentan en bloque. No se
usa para comparar: la nota de referencia es la de la lista con sorteo.

El sorteo no se eligió por dar la PR-AUC más alta, sino porque es el único desempate sin información añadida.
Desempatar por orden de fila no sería válido aunque diera mejor nota: ese orden depende de cómo se guardó el
CSV y puede estar relacionado con las variables del ingreso (en el apartado 3, `patient_nbr` se relaciona con
`number_diagnoses`, correlación de 0,225), así que no sería un desempate al azar.

El listón para la regresión logística del apartado 5 queda en PR-AUC 0,1978, ROC-AUC 0,6125 y recall en el
20 % 0,3496.